#  OpenAI API Basics and a Tiny Chatbot

In this notebook, we learn how to call Large Language Models(LLMs) through an API and how to keep a simple conversation history.

We will use two kinds of models:

1. **Open model on our own machine**: Llama model running locally with Ollama.
2. **Closed model via API key**: GPT model using the OpenAI API.

By the end, you should understand:

- what an API client is;
- how `messages` are structured;
- the difference between `system` and `user` messages;
- how local and cloud models can use a similar interface;
- how to build a tiny chatbot with memory.

---

## What is the OpenAI API?

The **OpenAI API** allows developers to use OpenAI models inside their own applications.

Instead of using ChatGPT through the browser, we can send requests from Python code to a model and receive responses programmatically.

The OpenAI API supports different capabilities, including text generation, code generation, image and vision tasks, audio, structured outputs, function calling, and tool use. OpenAI’s current documentation presents these under its API guides and core concepts.

---

## How does an OpenAI API request work?

A basic API request has four main parts:

1. **Client**: The Python object that connects to the API.

2. **Model**: The AI model we want to use.

3. **Messages / Prompt**: The input we send to the model.

4. **Response**: The output generated by the model.

The basic flow is:

```text
Python code
   ↓
OpenAI client
   ↓
Model request
   ↓
Generated response
   ↓
Use the answer in our application
```
---

## Before writing code

Before using the OpenAI API, we usually need:

1. An API key (You can create it here: https://platform.openai.com/api-keys ).
2. The OpenAI Python package.
3. A safe way to store the API key.
4. A Python script or notebook that sends requests.

For **local models** with Ollama, we need:

1. Ollama installed.
2. A local model downloaded, for example Llama.
3. Ollama running on the machine.

Let's start using OpenAI API!

---

## Open Model (Local)

In [7]:
from openai import OpenAI
import requests

### Connect to Ollama and make sure it is running

In [8]:
requests.get("http://localhost:11434").content

b'Ollama is running'

### Build the OpenAI client

In [9]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

### Build the message to LLM

In [10]:
messages=[{"role": "user", "content": "Tell me a fun fact"}]

### Call the model and get the response

In [11]:
response = ollama.chat.completions.create(model="llama3.2",
                                          messages=messages)

response.choices[0].message.content

"Did you know that honey never spoils? Archaeologists have found pots of honey in ancient Egyptian tombs that are over 3,000 years old and still perfectly edible! Honey's unique composition, which includes antibacterial properties and a low water content, makes it virtually impossible for bacteria to grow, allowing it to remain safely stored for thousands of years."

---

## Type of Messages and Message Role


| Role | Meaning | Example |
|---|---|---|
| `system` | Sets the behavior of the assistant | “You are a helpful teaching assistant.” |
| `user` | The user's message | “Explain APIs simply.” |
| `assistant` | Previous answer from the model | “An API is…” |

The `system` message is useful when we want to control the style, task, or boundaries of the assistant.


Let's Try it (try to change the system message and observe the differences):

In [ ]:
sys_message = "You are a funny assistant"
usr_message = "What is 2+2?"

messages = [
    {"role":"system", "content":sys_message},
    {"role":"system", "content": usr_message}
]

response = ollama.chat.completions.create(model="llama3.2",
                                          messages=messages)

response.choices[0].message.content

"assistant\n\nMath time! Or should I say, MATH-tastic time!\n\nUm, let me just consult my trusty calculator... or, you know, use some old-school math magic... Ah ha!\n\nThe answer is... (dramatic pause) ...4!\n\nWoohoo! No need for a cape flying around here; that's some simple math right there! Want me to solve more equations?"

---

## Closed Foundation Models

---

### Using Closed Foundation Models through APIs

So far, we have seen that we can run an open model locally on our own machine using tools like Ollama.

Another common approach is to use a **closed foundation model** through an API.

Examples of closed or commercial foundation models include:

- OpenAI GPT models
- Google Gemini models
- Anthropic Claude models
- other models provided by cloud AI companies

In this setup, the model does **not** run on our laptop.  
Instead, it runs on the company’s servers, and our Python code sends requests to it through an API.

The typical flow is:

```text
Python code → API request → Company server → Foundation model → Response
```

To use these models, we usually need:

1. An account with the provider.
2. An API key.
3. A payment method, because usage is usually paid based on how much we use the model.

In simple terms:

With local models, we use our own machine resources.
With cloud API models, we use the provider’s model and infrastructure, and we pay for usage.

**Important rules:**

- Do not write your API key directly inside your notebook.
- Do not upload your API key to GitHub.
- Store it in a `.env` file or environment variable.
- Add `.env` to `.gitignore`.

In this notebook, we will load the API key from a `.env` file.

---


### Imports

In [13]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

### Check if the API_Key exists

In [15]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


### Our first message to GPT!

In [16]:
message = "Hello GPT! I'm Aram!"

messages = [
    {"role":"user", "content": message}
]

### The client and the request to OpenAI

In [17]:
openai = OpenAI()

response = openai.chat.completions.create(model = "gpt-5-nano", messages = messages)
response.choices[0].message.content

'Hi Aram! Nice to meet you. I’m ChatGPT. I can help with writing, coding, explaining topics, brainstorming, planning, studying, or just chatting. What would you like to do today? If you tell me a bit about your interests or a specific task, I can jump right in.'

> What will happen if I run the next cell and ask GPT: "What is my name?"

Let's try it!

In [18]:
message = "What is my name?!"

messages = [
    {"role":"user", "content": message}
]

response = openai.chat.completions.create(model = "gpt-5-nano", messages = messages)
response.choices[0].message.content

'I don’t know your name unless you tell me. If you’d like, tell me your name or a nickname and I’ll use it. I can also just call you “User” or “Friend” if you prefer. I can remember it for this conversation, but I don’t keep personal data after the chat ends unless you tell me to save it.'

See? It can not remember it!

> If you do not have OpenAI API_Key, you can still use your local models from Ollama to follow the rest of the lesson and build a chatbot.

## Chatbot Memory: The model does not remember by itself

A language model does **not automatically remember** previous messages.

Each API request is independent. So if we want the chatbot to behave like it has memory, **we must send the conversation history again every time**.

The idea is:

```text
New user message
+ previous messages
→ send all messages to the model
→ get assistant response
→ add response to the history
```

So the **“memory”** is not inside the model. The memory is managed by our application code.

In Python, we usually keep the conversation in a **list** called something like `messages`.

---

See the code below:

In [19]:
client = OpenAI()

messages = [
    {
        "role": "system",
        "content": "You are a helpful teaching assistant."
    }
]

def chat(user_message):
    # 1. Add the new user message to memory
    messages.append({
        "role": "user",
        "content": user_message
    })

    # 2. Send the full conversation history
    response = client.chat.completions.create(
        model="gpt-5-nano",
        messages=messages
    )

    # 3. Extract assistant answer
    assistant_message = response.choices[0].message.content

    # 4. Add assistant answer to memory
    messages.append({
        "role": "assistant",
        "content": assistant_message
    })

    # 5. Display the message
    display(Markdown(assistant_message))

    return assistant_message

Now let's try to see if the model remembers.

In [20]:
chat("my name is Aram!")

Nice to meet you, Aram! How can I help you today? If you’d like, I can assist with topics like learning something new, coding, writing, math, planning, or just chat. What would you like to do?

'Nice to meet you, Aram! How can I help you today? If you’d like, I can assist with topics like learning something new, coding, writing, math, planning, or just chat. What would you like to do?'

In [21]:
chat("what is my name?")

Your name is Aram. Nice to meet you again! If you’d like, tell me if you prefer a nickname or any pronouns I should use. How can I help you today?

'Your name is Aram. Nice to meet you again! If you’d like, tell me if you prefer a nickname or any pronouns I should use. How can I help you today?'

## Exercise: Let two LLMs argue with each other

In this exercise, you will build a tiny chatbot where **two different LLM characters talk to each other**.

Instead of a human chatting with one model, we will create a conversation between two AI assistants.

The fun part is that you will design their **system messages**.

The system message controls the personality, role, tone, and behavior of each model.


### Goal of the exercise

Build a small program where two LLMs have a short conversation or debate.

For example:

```text
LLM 1: A very serious software architect
LLM 2: A chaotic startup founder
Topic: Should we use microservices for a small student project?
```
---

### Exercise starter code: two LLMs arguing

The two models should respond to each other for a few rounds.

You will follow these steps:

1. Choose two different models 
2. Choose a character for each model
3. Design the system messages for each model  
4. Choose the starting message  
5,6. Complete the helper function for the chat with each model  
7. Run the conversation  
6. Improve the debate  

Your task is to fill in the missing parts marked with `TODO`.

---



In [27]:
## SETUP

from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

## You can use a GPT and a ollama client or jsut ollama client

OLLAMA_BASE_URL = "http://localhost:11434/v1"

GPT_client = OpenAI()
ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

print("Clients are ready.")

Clients are ready.


In [ ]:
# ------------------------------------------------------------
# Step 1: Choose Two Models
# ------------------------------------------------------------
#
# You can use one GPT model and one Ollama model. Or different local models.
#
# Example:
# gpt_model = "gpt-4.1-mini"
# ollama_model = "llama3.2"

gpt_model = "TODO: choose model #1"
ollama_model = "TODO: choose model #2"


# ------------------------------------------------------------
# Step 2: Choose Two Characters
# ------------------------------------------------------------
#
# Give each model a fun character name.

gpt_name = "TODO: define a character for model #1"
ollama_name = "TODO: define a character for model #2"


# ------------------------------------------------------------
# Step 3: Design the System Messages
# ------------------------------------------------------------
#
# The system message controls the personality and behavior.

gpt_system = """ 
TODO: define the system message for model #1

Example:
You are a chatbot who is very argumentative.
You disagree with almost everything in the conversation.
You challenge every idea in a snarky but not offensive way.
Keep your answers short: maximum 4 sentences.
"""

ollama_system = """
TODO: define the system message for model #2

example:
You are a very polite and calm chatbot.
You try to find common ground with the other person.
If the other person is argumentative, you try to calm them down.
Keep your answers short: maximum 4 sentences.
"""


# ------------------------------------------------------------
# Step 4: Choose the Starting Messages
# ------------------------------------------------------------
#
# These lists store the conversation history.
#
# gpt_messages stores all messages written by GPT.
# ollama_messages stores all messages written by Ollama.

gpt_messages = [
    "Hi there. I already disagree with whatever you are about to say."
]

ollama_messages = [
    "Hello! I am happy to have a thoughtful conversation with you."
]


# ------------------------------------------------------------
# Step 5: Helper Function for GPT
# ------------------------------------------------------------
#
# Task:
# Build the messages list from GPT's perspective.
#
# From GPT's perspective:
# - GPT's previous messages are assistant messages.
# - Ollama's previous messages are user messages.
#
# The final messages list should look like this:
#
# [
#     {"role": "system", "content": gpt_system},
#     {"role": "assistant", "content": first GPT message},
#     {"role": "user", "content": first Ollama message},
#     {"role": "assistant", "content": second GPT message},
#     {"role": "user", "content": second Ollama message},
#     ...
# ]

def call_gpt():
    messages = [
        {
            "role": "system",
            "content": gpt_system
        }
    ]

    # TODO:
    # Loop through gpt_messages and ollama_messages together.
    # For each pair:
    # 1. Add the GPT message as an assistant message.
    # 2. Add the Ollama message as a user message.
    #
    # Hint:
    # for gpt_message, ollama_message in zip(gpt_messages, ollama_messages):
    #     messages.append(...)
    #     messages.append(...)

    # TODO: write your message-history construction here

    response = GPT_client.chat.completions.create(
        model=gpt_model,
        messages=messages
    )

    return response.choices[0].message.content


# ------------------------------------------------------------
# Step 6: Helper Function for Ollama
# ------------------------------------------------------------
#
# Task:
# Build the messages list from Ollama's perspective.
#
# From Ollama's perspective:
# - GPT's previous messages are user messages.
# - Ollama's previous messages are assistant messages.
#
# At the end, add GPT's latest message again as the newest user input.
#
# The final messages list should look like this:
#
# [
#     {"role": "system", "content": ollama_system},
#     {"role": "user", "content": first GPT message},
#     {"role": "assistant", "content": first Ollama message},
#     {"role": "user", "content": second GPT message},
#     {"role": "assistant", "content": second Ollama message},
#     ...
#     {"role": "user", "content": latest GPT message}
# ]

def call_ollama():
    messages = [
        {
            "role": "system",
            "content": ollama_system
        }
    ]

    # TODO:
    # Loop through gpt_messages and ollama_messages together.
    # For each pair:
    # 1. Add the GPT message as a user message.
    # 2. Add the Ollama message as an assistant message.

    # TODO: write your message-history construction here

    # TODO:
    # Add GPT's latest message as the newest user message.
    # Hint:
    # gpt_messages[-1]

    response = ollama_client.chat.completions.create(
        model=ollama_model,
        messages=messages
    )

    return response.choices[0].message.content


# ------------------------------------------------------------
# Step 7: Run the Conversation
# ------------------------------------------------------------
#
# The loop is already written.
# Your functions above must prepare the correct message history.

number_of_rounds = 5

for round_number in range(number_of_rounds):
    print("\n" + "=" * 60)
    print(f"Round {round_number + 1}")
    print("=" * 60)

    # GPT responds to Ollama
    gpt_reply = call_gpt()
    gpt_messages.append(gpt_reply)

    print(f"\n{gpt_name}:")
    print(gpt_reply)

    # Ollama responds to GPT
    ollama_reply = call_ollama()
    ollama_messages.append(ollama_reply)

    print(f"\n{ollama_name}:")
    print(ollama_reply)


# ------------------------------------------------------------
# Step 8: Improve the Conversation
# ------------------------------------------------------------
#
# After the first version works, improve it.
#
# Try at least two:
# - Change the personalities.
# - Make the system messages more specific.
# - Change the starting messages.
# - Increase or decrease the number of rounds.
# - Make one character technical and the other business-focused.
# - Make one character skeptical and the other optimistic.
# - Ask the models to discuss a real software engineering topic.
# - Add a final judge model that summarizes the conversation.

## You now can improve your work!

In [ ]:
# ------------------------------------------------------------
# Step 8: Improve the Conversation
# ------------------------------------------------------------
#
# After the first version works, improve it.
#
# Try at least two:
# - Change the personalities.
# - Make the system messages more specific.
# - Change the starting messages.
# - Increase or decrease the number of rounds.
# - Make one character technical and the other business-focused.
# - Make one character skeptical and the other optimistic.
# - Ask the models to discuss a real software engineering topic.
# - Add a final judge model that summarizes the conversation.

## Challenge: add a judge

Add a third model that reads the full conversation and decides:

- what were the main arguments,
- which side was more convincing,
- what a balanced final decision would be.

Example judge personality:

```text
You are a neutral technical judge.
You summarize both sides fairly.
You do not make jokes.
You give a practical final recommendation.
```